# Day 1 — NLP Preprocessing

## IMDb Movie Reviews

In this notebook, we explore the fundamental steps of Natural Language Processing (NLP) preprocessing using the IMDb Movie Reviews dataset.

The goal is to understand how raw text can be cleaned, normalized, and transformed into a consistent form before being used in an NLP task.

We will focus on:

- Tokenization
- Lowercasing
- Punctuation removal
- Stop-word removal
- Lemmatization
- Preserving task-critical information such as negations

In [1]:
import pandas as pd
import numpy as np
import re
import string

import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer

In [5]:
import pandas as pd

df = pd.read_csv(
    "IMDB Dataset.csv",
    on_bad_lines="skip"
)

print(df.shape)
df.head()

(50000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [7]:
df.isnull().sum()

,0
review,0
sentiment,0


In [8]:
df['sentiment'].value_counts()

,count
sentiment,
positive,25000
negative,25000


##  Tokenization

Tokenization is the process of splitting text into smaller units called **tokens**.

For this project, we will use **word-level tokenization**, where each word or punctuation mark is treated as a separate token.

Tokenization is an important first step in NLP preprocessing because it allows us to work with individual words instead of treating the entire review as one text string.

In this step, we will:
- Select a sample review from the dataset.
- Apply word tokenization using NLTK.
- Compare the original text with the resulting tokens.

In [10]:
text = df['review'].iloc[0]

tokens = word_tokenize(text)

print("Original text:")
print(text)

print("\nTokens:")
print(tokens)

Original text:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is

### Tokenization Results

The tokenization step successfully split the raw review into individual tokens.

The output shows that tokens can include:

- Individual words such as `One`, `reviewers`, `watching`, and `violence`.
- Numbers such as `1`.
- Punctuation marks such as `.`, `,`, and `(`.
- HTML tags such as `<br />`, which appear as separate tokens.
- Contractions such as `you'll`, `wouldn't`, and `couldn't`, which are split into multiple tokens.

### Observations

The tokenization result shows that the raw text still contains unnecessary elements that should be handled during preprocessing.

For example:

- HTML tags such as `<br />` should be removed.
- Punctuation marks should be removed when they are not useful for the task.
- Numbers may be removed depending on their importance.
- Stop words can be removed to reduce unnecessary words.
- Negation words such as `not`, `no`, and `never` should be preserved because they can strongly affect sentiment.
- Contractions such as `wouldn't` and `couldn't` should be handled carefully because removing their negation part can change the meaning of the review.

Therefore, tokenization is only the first step. The text needs additional cleaning and normalization before it can be used for further NLP processing.

##  Lowercasing

Lowercasing converts all alphabetic characters in the text to lowercase.

For example:

`The Movie Was AMAZING`

becomes:

`the movie was amazing`

Lowercasing helps treat words with different capitalization as the same word.

For example:

`Movie`, `movie`, and `MOVIE`

should generally be considered the same word during text processing.

However, capitalization can sometimes carry meaning, so the choice to lowercase depends on the NLP task. For this sentiment analysis task, lowercasing is appropriate because we are mainly interested in the meaning of the words rather than their capitalization.

In [11]:
text = df['review'].iloc[0]

lower_text = text.lower()

print("Original text:")
print(text[:500])

print("\nLowercased text:")
print(lower_text[:500])

Original text:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ

Lowercased text:
one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.<br /><br />the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this is not a show for the faint hearted or timid. this show pulls no punches with regards to drugs, sex or violence. its is hardcore, in the classic use of th

## Removing HTML Tags and Punctuation

The raw IMDb reviews contain HTML tags such as `<br />`, which were originally used to represent line breaks in the reviews.

These tags do not provide useful information for sentiment analysis, so they should be removed.

The reviews also contain punctuation marks such as periods, commas, brackets, and other symbols.

For this preprocessing task, punctuation will be removed to simplify the text and focus on the words.

However, punctuation and symbols can sometimes carry useful information in sentiment analysis. Therefore, this choice should be documented and may be reconsidered when building the final sentiment model.

In [13]:
import re

no_html = re.sub(r'<.*?>', ' ', lower_text)

print("Before removing HTML:")
print(lower_text[:500])

print("\nAfter removing HTML:")
print(no_html[:500])

Before removing HTML:
one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.<br /><br />the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this is not a show for the faint hearted or timid. this show pulls no punches with regards to drugs, sex or violence. its is hardcore, in the classic use of the word.<br /><br />it is called oz

After removing HTML:
one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.  the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this is not a show for the faint hearted or timid. this show pulls no punches with regards to drugs, sex or violence. its is hardcore, in the classic use of t

##  Removing Punctuation

Punctuation marks such as `.`, `,`, `!`, `?`, `(`, and `)` are removed to simplify the text.

For this preprocessing task, punctuation is not considered necessary for the basic sentiment analysis pipeline, so removing it helps reduce unnecessary tokens and vocabulary variation.

In [14]:
no_punctuation = no_html.translate(
    str.maketrans('', '', string.punctuation)
)

print("Before removing punctuation:")
print(no_html[:500])

print("\nAfter removing punctuation:")
print(no_punctuation[:500])

Before removing punctuation:
one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.  the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this is not a show for the faint hearted or timid. this show pulls no punches with regards to drugs, sex or violence. its is hardcore, in the classic use of the word.  it is called oz as that is the nick

After removing punctuation:
one of the other reviewers has mentioned that after watching just 1 oz episode youll be hooked they are right as this is exactly what happened with me  the first thing that struck me about oz was its brutality and unflinching scenes of violence which set in right from the word go trust me this is not a show for the faint hearted or timid this show pulls no punches with regards to drugs sex or violence its is hardcore in the classic use o

##  Stop-word Removal

Stop words are common words that appear frequently in a language but may provide limited useful information for some NLP tasks.

Examples of common English stop words include:

- `the`
- `is`
- `and`
- `of`
- `to`
- `in`
- `a`

Removing stop words can reduce the number of tokens and simplify the text.

However, for sentiment analysis, some words that are commonly classified as stop words can be very important. In particular, negation words such as:

- `not`
- `no`
- `never`

can completely change the meaning of a sentence.

For this reason, these negation words will be preserved instead of being removed.

In [15]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

print("Number of stop words:", len(stop_words))
print("Sample stop words:")
print(list(stop_words)[:20])

Number of stop words: 198
Sample stop words:
['both', 'from', 'then', 'whom', 'ma', 'him', 'she', 'which', 'hers', 'at', 'doesn', 'up', 'i', 'those', 've', "don't", "should've", 'why', 'its', 'so']


In [18]:
important_negations = {"not", "no", "never"}

stop_words_without_negations = stop_words - important_negations

print("Preserved negations:", important_negations)

Preserved negations: {'never', 'no', 'not'}


In [19]:
tokens = word_tokenize(no_punctuation)

filtered_tokens = [
    word for word in tokens
    if word not in stop_words_without_negations
]

print("Before stop-word removal:")
print(tokens[:50])

print("\nAfter stop-word removal:")
print(filtered_tokens[:50])

Before stop-word removal:
['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', '1', 'oz', 'episode', 'youll', 'be', 'hooked', 'they', 'are', 'right', 'as', 'this', 'is', 'exactly', 'what', 'happened', 'with', 'me', 'the', 'first', 'thing', 'that', 'struck', 'me', 'about', 'oz', 'was', 'its', 'brutality', 'and', 'unflinching', 'scenes', 'of', 'violence', 'which', 'set', 'in', 'right', 'from', 'the']

After stop-word removal:
['one', 'reviewers', 'mentioned', 'watching', '1', 'oz', 'episode', 'youll', 'hooked', 'right', 'exactly', 'happened', 'first', 'thing', 'struck', 'oz', 'brutality', 'unflinching', 'scenes', 'violence', 'set', 'right', 'word', 'go', 'trust', 'not', 'show', 'faint', 'hearted', 'timid', 'show', 'pulls', 'no', 'punches', 'regards', 'drugs', 'sex', 'violence', 'hardcore', 'classic', 'use', 'word', 'called', 'oz', 'nickname', 'given', 'oswald', 'maximum', 'security', 'state']


##  Lemmatization

Lemmatization is the process of reducing words to their **base or dictionary form**, known as a lemma.

Unlike simple word truncation, lemmatization considers the actual form and meaning of a word.

For example:

- `running` → `running`
- `runs` → `run`
- `studies` → `study`
- `better` → `better`

Lemmatization generally produces more meaningful results than stemming because it aims to return valid dictionary words.

For this project, we will use NLTK's `WordNetLemmatizer`.

In [21]:
lemmatizer = WordNetLemmatizer()

lemmatized_tokens = [
    lemmatizer.lemmatize(word)
    for word in filtered_tokens
]

print("Before lemmatization:")
print(filtered_tokens[:50])

print("\nAfter lemmatization:")
print(lemmatized_tokens[:50])


Before lemmatization:
['one', 'reviewers', 'mentioned', 'watching', '1', 'oz', 'episode', 'youll', 'hooked', 'right', 'exactly', 'happened', 'first', 'thing', 'struck', 'oz', 'brutality', 'unflinching', 'scenes', 'violence', 'set', 'right', 'word', 'go', 'trust', 'not', 'show', 'faint', 'hearted', 'timid', 'show', 'pulls', 'no', 'punches', 'regards', 'drugs', 'sex', 'violence', 'hardcore', 'classic', 'use', 'word', 'called', 'oz', 'nickname', 'given', 'oswald', 'maximum', 'security', 'state']

After lemmatization:
['one', 'reviewer', 'mentioned', 'watching', '1', 'oz', 'episode', 'youll', 'hooked', 'right', 'exactly', 'happened', 'first', 'thing', 'struck', 'oz', 'brutality', 'unflinching', 'scene', 'violence', 'set', 'right', 'word', 'go', 'trust', 'not', 'show', 'faint', 'hearted', 'timid', 'show', 'pull', 'no', 'punch', 'regard', 'drug', 'sex', 'violence', 'hardcore', 'classic', 'use', 'word', 'called', 'oz', 'nickname', 'given', 'oswald', 'maximum', 'security', 'state']


##  Stemming vs. Lemmatization

Both stemming and lemmatization aim to reduce words to a simpler form, but they work differently.

### Stemming

Stemming removes prefixes or suffixes using simple rules. The resulting word may not always be a valid dictionary word.

For example:

`studies` → `studi`

### Lemmatization

Lemmatization uses linguistic information to return a meaningful base or dictionary form.

For example:

`studies` → `study`

### Comparison

| Method | Approach | Output | Linguistic quality |
|--------|----------|--------|--------------------|
| Stemming | Removes word endings | May not be a real word | Lower |
| Lemmatization | Uses vocabulary and linguistic rules | Usually a valid word | Higher |

For this project, **Lemmatization** is preferred because preserving meaningful words is useful for sentiment analysis and makes the preprocessing results easier to interpret.

##  Complete Text Preprocessing Pipeline

So far, we have applied each preprocessing step separately:

1. Lowercasing
2. HTML tag removal
3. Punctuation removal
4. Tokenization
5. Stop-word removal
6. Negation preservation
7. Lemmatization

Now, these steps will be combined into a single preprocessing function.

Creating a preprocessing pipeline makes the process consistent and reusable across the entire dataset.

For sentiment analysis, special attention will be given to negation words such as `not`, `no`, and `never`, which will be preserved.

In [22]:
def preprocess_text(text):
    # 1. Lowercase
    text = text.lower()

    # 2. Remove HTML tags and replace them with spaces
    text = re.sub(r'<.*?>', ' ', text)

    # 3. Tokenize
    tokens = word_tokenize(text)

    # 4. Remove punctuation and numbers
    tokens = [
        word for word in tokens
        if word not in string.punctuation and not word.isdigit()
    ]

    # 5. Remove stop words while preserving negations
    important_negations = {"not", "no", "never"}

    tokens = [
        word for word in tokens
        if word not in stop_words or word in important_negations
    ]

    # 6. Lemmatization
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    return tokens

In [23]:
sample_text = df['review'].iloc[0]

processed_text = preprocess_text(sample_text)

print("Original review:")
print(sample_text[:500])

print("\nProcessed tokens:")
print(processed_text[:50])

Original review:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ

Processed tokens:
['one', 'reviewer', 'mentioned', 'watching', 'oz', 'episode', "'ll", 'hooked', 'right', 'exactly', 'happened', 'first', 'thing', 'struck', 'oz', 'brutality', 'unflinching', 'scene', 'violence', 'set', 'right', 'word', 'go', 'trust', 'not', 'show', 'faint', 'hearted', 'timid', 'show', 'pull', 'no', 'punch', 'regard', 'drug', 'sex', 'violence', 'hardcore', 'classic', 'use', 'word', 'called', 'oz', 'nickname', 'given', 'oswald', 'maximum', 'security', 'state', '

##  Before vs. After Preprocessing

To evaluate the effect of preprocessing, we will compare the number of tokens before and after cleaning.

This comparison helps us understand how much unnecessary information was removed from the original review.

In [24]:
original_tokens = word_tokenize(sample_text)

print("Number of tokens before preprocessing:", len(original_tokens))
print("Number of tokens after preprocessing:", len(processed_text))
print("Tokens removed:", len(original_tokens) - len(processed_text))

Number of tokens before preprocessing: 380
Number of tokens after preprocessing: 178
Tokens removed: 202


In [25]:
negation_test = "I did not like this movie at all. I never recommend it."

processed_negation = preprocess_text(negation_test)

print("Original text:")
print(negation_test)

print("\nProcessed tokens:")
print(processed_negation)

print("\nNegations found:")
for word in ["not", "no", "never"]:
    print(f"{word}: {word in processed_negation}")

Original text:
I did not like this movie at all. I never recommend it.

Processed tokens:
['not', 'like', 'movie', 'never', 'recommend']

Negations found:
not: True
no: False
never: True


##  Preprocessing Summary

In this notebook, several NLP preprocessing techniques were applied to the IMDb movie review dataset.

The preprocessing workflow included:

1. **Lowercasing**  
   Converted all text to lowercase to reduce vocabulary variation.

2. **HTML Removal**  
   Removed HTML tags such as `<br />` from the reviews.

3. **Tokenization**  
   Split each review into individual tokens.

4. **Punctuation Removal**  
   Removed punctuation marks that were not required for the basic preprocessing pipeline.

5. **Stop-word Removal**  
   Removed common words that provide limited information for the task.

6. **Negation Preservation**  
   Preserved important negation words such as `not`, `no`, and `never`.

7. **Lemmatization**  
   Reduced words to their base forms where possible.

### Key Result

For the sample review:

- Tokens before preprocessing: **380**
- Tokens after preprocessing: **178**
- Tokens removed: **202**
- Reduction: **53.16%**

The preprocessing process produced a cleaner and more compact representation of the original text while attempting to preserve sentiment-critical information.

##  Conclusion

In this notebook, we explored the main steps required to preprocess raw text for NLP tasks using the IMDb Movie Reviews dataset.

We learned how to:

- Inspect and understand a text dataset.
- Tokenize raw text into individual words.
- Convert text to lowercase.
- Remove unnecessary HTML tags.
- Remove punctuation and standalone numbers.
- Remove common stop words.
- Preserve important negation words such as `not`, `no`, and `never`.
- Apply lemmatization to reduce words to their base forms.
- Compare the text before and after preprocessing.
- Identify preprocessing limitations, especially when handling contractions.

The preprocessing experiment showed that the sample review was reduced from **380 tokens to 178 tokens**, removing **202 tokens** and reducing the token count by approximately **53.16%**.

The main lesson is that NLP preprocessing is not simply about removing as much text as possible. Preprocessing decisions should depend on the task. For sentiment analysis, important linguistic information such as negation must be preserved.

The resulting preprocessing pipeline provides a cleaner representation of the reviews and prepares the text for future NLP tasks such as feature extraction, text classification, and sentiment analysis.